# Exercise — Galaxy Clustering and Power Spectrum from the Magneticum Hydrodynamical Simulation

**Course topics covered:**
- Loading and cleaning a large-scale simulation galaxy catalogue.
- Visualising the three-dimensional large-scale structure and the cosmic web.
- Computing the stellar mass function and star-formation scaling relations.
- Implementing a galaxy power-spectrum estimator from scratch (NGP → FFT → spherical binning).
- Measuring galaxy bias as a function of stellar mass and star-formation activity.
- *(Optional)* Comparing mass-assignment schemes: NGP, CIC, TSC.

**Reference papers:**
- Dolag, K., et al. 2016, *MNRAS* **463**, L69 — overview of the Magneticum Pathfinder simulations.
  Web portal and data access: <http://www.magneticum.org>
- Springel, V., et al. 2005, *Nature* **435**, 629 — the Millennium simulation (N-body counterpart).
- Eisenstein, D. J. & Hu, W. 1998, *ApJ* **496**, 605 — analytical CDM transfer function.
- Tinker, J. L., et al. 2010, *ApJ* **724**, 878 — halo bias fitting function.
- Croton, D. J., et al. 2007, *MNRAS* **374**, 1303 — galaxy clustering and bias.

**Dataset:** Magneticum Pathfinder **Box2b/hr** galaxy catalogue, snapshot 031
(`galaxies.txt.gz`, ~547 MB compressed).
Download from <http://wwwmpa.mpa-garching.mpg.de/HydroSims/Magneticum/Downloads/Magneticum/Box2b_hr/snap_031/>.

Box size $L = 640\;h^{-1}\,\mathrm{Mpc}$, cosmology WMAP7
($\Omega_m=0.272$, $\Omega_\Lambda=0.728$, $h=0.704$, $\sigma_8=0.809$, $n_s=0.963$).

**Catalogue columns** (after loading):

| Name | Description | Unit |
|------|-------------|------|
| `uid` | Unique galaxy identifier | — |
| `x_kpc`, `y_kpc`, `z_kpc` | Comoving position | $h^{-1}$ kpc |
| `m_star` | Stellar mass | $h^{-1}\,M_\odot$ |
| `sfr` | Star formation rate | $M_\odot\,\mathrm{yr}^{-1}$ |
| `host` | UID of the group's central galaxy | — |
| `dist_kpc` | Distance to group centre | $h^{-1}$ kpc |
| `log_m_cD` | $\log_{10}(m_{\rm cD}/m_*)$ | — |
| `m_gas` | Stellar mass of the gas component | $h^{-1}\,M_\odot$ |
| `vx`, `vy`, `vz` | Peculiar velocity components | km s$^{-1}$ |

**Scientific scenario.** Hydrodynamical cosmological simulations model dark matter, gas, stars,
and black holes self-consistently. Unlike pure N-body runs, they produce realistic galaxies
with stellar masses, star-formation rates, and ICM properties. The spatial distribution of
these galaxies is a biased tracer of the underlying matter field: the **galaxy power spectrum**
$P_{\rm gal}(k) = b^2\,P_{\rm matter}(k)$ on large scales, where the bias $b$ depends on the
galaxy sample selected. Comparing $P(k)$ for star-forming versus quiescent galaxies, or for
different stellar-mass thresholds, directly probes the environmental dependence of galaxy
formation.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import linregress
import os

# Add any additional imports you need

BOX_SIZE = 640.0   # h^-1 Mpc
CAT_FILE = "galaxies.txt.gz"


## Task 0 — Download, load, and inspect the catalogue

The file `galaxies.txt.gz` can be downloaded from:

```
http://wwwmpa.mpa-garching.mpg.de/HydroSims/Magneticum/Downloads/Magneticum/Box2b_hr/snap_031/
```

or with `wget`:

```bash
wget http://wwwmpa.mpa-garching.mpg.de/HydroSims/Magneticum/Downloads/Magneticum/Box2b_hr/snap_031/galaxies.txt.gz
```

The file is **space-separated ASCII** with a single comment header line (starting with `#`).
Loading ~9 million rows takes 1–2 minutes; consider using `nrows=100_000` for quick exploration.

- How many galaxies are in the catalogue? What is the stellar mass range?
- What fraction of galaxies have $\mathrm{SFR} > 0$ (star-forming)?
- How are galaxies classified as centrals vs. satellites? *Hint: look at `dist_kpc`.*
- What is the minimum stellar mass in the catalogue?
  Is the catalogue complete down to this mass, or is there a more conservative limit?


In [ ]:
COL_NAMES = ['uid', 'x_kpc', 'y_kpc', 'z_kpc', 'x_pix', 'y_pix', 'i_sector',
             'm_star', 'sfr', 'host', 'dist_kpc', 'log_m_cD', 'm_gas',
             'vx', 'vy', 'vz', 'dv', 'vr', 'vt']

# YOUR CODE HERE
# 1. Load galaxies.txt.gz with pd.read_csv (sep=r'\s+', comment='#', header=None, names=COL_NAMES).
# 2. Add derived columns: x, y, z in h^-1 Mpc (divide kpc by 1000);
#    log10_ms = log10(m_star); is_sf = (sfr > 0); is_central = (dist_kpc < 1).
# 3. Print: number of galaxies, mass range, star-forming fraction, central fraction.


## Task 1 — Three-dimensional spatial distribution

- Plot the three 2D projections of galaxy positions (XY, XZ, YZ),
  colour-coded by $\log_{10} M_*$.
  Use a random subsample of $\sim 10^5$ galaxies for clarity.
- Plot a thin slice ($\Delta z < 30\;h^{-1}\,\mathrm{Mpc}$) to highlight filaments, walls, and voids.
- Overplot star-forming and quiescent galaxies in different colours on the thin-slice projection.
  Do the two populations trace the same structures? Are quiescent galaxies more concentrated
  in dense regions?
- Estimate the mean inter-galaxy separation and compare with the typical size of a galaxy group.


In [ ]:
# YOUR CODE HERE
# 1. Draw a random subsample of ~100,000 galaxies for plotting.
# 2. Plot the three 2D projections colour-coded by log10(m_star).
# 3. Plot a thin slice (z < 30 h^-1 Mpc), distinguishing SF and quiescent galaxies.
# 4. Print the mean number density and mean inter-galaxy separation.


## Task 2 — Galaxy power spectrum from scratch

### Algorithm outline

1. **Mass assignment — Nearest Grid Point (NGP)**
   Each galaxy is assigned to the single grid cell that contains it.
   The grid has $N^3$ cells of side $\Delta x = L/N$.

2. **Overdensity field**
   $$\delta(\mathbf{x}) = \frac{n(\mathbf{x})}{\bar{n}} - 1, \qquad \bar{n} = \frac{N_{\rm gal}}{N^3}$$

3. **Fourier transform** (using `np.fft.rfftn`)
   $$\tilde{\delta}(\mathbf{k}) = \sum_{\mathbf{x}} \delta(\mathbf{x})\,e^{-i\mathbf{k}\cdot\mathbf{x}}$$

4. **Power spectrum estimator**
   $$P(k) = \frac{L^3}{N^6}\left|\tilde{\delta}(\mathbf{k})\right|^2 \quad \text{averaged over modes with } |\mathbf{k}|\approx k$$

5. **Shot-noise subtraction**
   $$P_{\rm shot} = \frac{V}{N_{\rm gal}} = \frac{L^3}{N_{\rm gal}}$$

Key scales:
- Fundamental mode: $k_{\rm fund} = 2\pi/L$
- Nyquist frequency: $k_{\rm Nyq} = \pi N/L$

Questions:
- Why must we subtract shot noise? What would the uncorrected $P(k)$ look like at large $k$?
- At which $k$ does the halo power spectrum exceed the linear theory? What causes this?
- The BAO peak at $k \approx 0.06\;h\,\mathrm{Mpc}^{-1}$ — can you see it in the simulation?


In [ ]:
def ngp_assign(positions, box_size, n_grid):
    """
    Nearest Grid Point mass assignment.

    Parameters
    ----------
    positions : (N, 3) float array — comoving positions in h^-1 Mpc, range [0, box_size)
    box_size  : float — box side in h^-1 Mpc
    n_grid    : int   — grid cells per dimension

    Returns
    -------
    grid : (n_grid, n_grid, n_grid) float array — galaxy count field
    """
    # YOUR CODE HERE
    # Steps:
    #   1. cell_size = box_size / n_grid
    #   2. Integer grid indices: ix = floor(x / cell_size) % n_grid  (and same for y, z)
    #   3. np.add.at(grid, (ix, iy, iz), 1)
    pass


In [ ]:
def pk_from_grid(grid, box_size, n_halos=None, subtract_shot_noise=True):
    """
    Estimate the power spectrum from a 3-D count grid.

    Parameters
    ----------
    grid    : (N, N, N) array — galaxy count per cell
    box_size: float           — box side in h^-1 Mpc
    n_halos : int or None     — total galaxy count (for shot-noise subtraction)
    subtract_shot_noise : bool

    Returns
    -------
    k  : 1-D array  — shell-averaged wavenumber in h Mpc^-1
    Pk : 1-D array  — power spectrum in (h^-1 Mpc)^3
    Nk : 1-D array  — number of modes per shell
    """
    n_grid = grid.shape[0]

    # YOUR CODE HERE
    # Steps:
    #   1. delta = grid / grid.mean() - 1
    #   2. delta_k = np.fft.rfftn(delta)
    #   3. Pk_3d = (box_size**3 / n_grid**6) * np.abs(delta_k)**2
    #   4. Build 3-D K array using np.fft.fftfreq / rfftfreq:
    #        k_j = fftfreq(n_grid, d=box_size/n_grid) * 2*pi  (in h/Mpc)
    #   5. Bin Pk_3d into spherical shells with np.histogram; exclude K=0
    #   6. Subtract box_size**3 / n_halos if requested
    pass


def compute_pk(positions, box_size, n_grid=256, assign_func=None):
    """Convenience wrapper: positions array -> (k, Pk, Nk)."""
    if assign_func is None:
        assign_func = ngp_assign
    grid = assign_func(positions, box_size, n_grid)
    return pk_from_grid(grid, box_size, n_halos=len(positions))


In [ ]:
MIN_MASS = 1e9    # h^-1 Msun
N_GRID   = 256
# YOUR CODE HERE
# 1. Select galaxies with m_star > MIN_MASS and extract positions as a (N,3) array.
# 2. Build the NGP density grid and compute delta = grid/mean - 1.
# 3. Plot a 2-D slice through the middle of the box.
# 4. Plot the 1-D distribution of delta values.


In [ ]:
# Eisenstein & Hu (1998) transfer function — provided, do not modify

def _eh98_transfer(k, Omega_m=0.272, Omega_b=0.0456, h=0.704, T_cmb=2.7255):
    ombh2 = Omega_b * h**2;  ommh2 = Omega_m * h**2
    k_eq  = 7.46e-2 * ommh2 * (T_cmb / 2.7)**(-2)
    b1    = 0.313 * ommh2**(-0.419) * (1 + 0.607 * ommh2**0.674)
    b2    = 0.238 * ommh2**0.223
    z_d   = 1291 * ommh2**0.251 / (1 + 0.659 * ommh2**0.828) * (1 + b1 * ombh2**b2)
    alpha_c = (1 - 0.328 * np.log(431 * ommh2) * ombh2 / Omega_m
               + 0.38 * np.log(22.3 * ommh2) * (ombh2 / Omega_m)**2)
    beta_c  = 1 / (1 - 0.949 * ombh2 / Omega_m)**(-0.419
               * np.log(431 * ommh2) * ombh2 / Omega_m)
    q  = k / (13.41 * k_eq)
    C  = 14.2 / alpha_c + 386 / (1 + 69.9 * q**1.08)
    return np.log(np.e + 1.8*beta_c*q) / (np.log(np.e + 1.8*beta_c*q) + C*q**2)

def linear_pk_eh98(k, n_s=0.963, sigma8=0.809, Omega_m=0.272, Omega_b=0.0456, h=0.704):
    T   = _eh98_transfer(k, Omega_m, Omega_b, h)
    Pk  = k**n_s * T**2
    R8  = 8.0
    k_s = np.logspace(-4, 3, 2000)
    T_s = _eh98_transfer(k_s, Omega_m, Omega_b, h)
    W   = 3 * (np.sin(k_s*R8) - k_s*R8*np.cos(k_s*R8)) / (k_s*R8)**3
    sigma2 = np.trapz(k_s**n_s * T_s**2 * W**2 * k_s**2, k_s) / (2*np.pi**2)
    return sigma8**2 / sigma2 * Pk


# YOUR CODE HERE
# 1. Compute P(k) for galaxies with M* > 10^9 h^-1 Msun using compute_pk with N_GRID=256.
# 2. Compute linear P(k) with linear_pk_eh98.
# 3. Plot both on a log-log scale; mark the Nyquist frequency.


## Task 3 — Galaxy bias: mass cuts and star-formation activity

On large scales ($k \lesssim 0.1\;h\,\mathrm{Mpc}^{-1}$) galaxy clustering obeys

$$P_{\rm gal}(k) \approx b^2\,P_{\rm matter}(k)$$

where the linear bias $b$ depends on how galaxies are selected. Massive, red (quiescent)
galaxies preferentially inhabit dense environments and are therefore more biased
($b > 1$) relative to the mean matter distribution. Low-mass, blue (star-forming)
galaxies trace the matter field more faithfully ($b \lesssim 1$).

**Part A — Stellar mass cuts:**
Compute $P(k)$ for galaxies above the following stellar mass thresholds and overplot:
$$M_* > \{10^9,\;10^{10},\;10^{10.5},\;10^{11}\}\;h^{-1}M_\odot$$

**Part B — Star-forming vs. quiescent:**
Compute $P(k)$ separately for star-forming (SFR $> 0$) and quiescent (SFR $= 0$) galaxies.

For each subsample:
- Estimate the large-scale bias $b = \sqrt{P_{\rm gal}/P_{\rm lin}}$ averaged over
  $k < 0.1\;h\,\mathrm{Mpc}^{-1}$.
- Plot $b(k)$ vs. $k$ to check for scale dependence.
- Are the bias values consistent with the expectation that more massive / more quiescent
  galaxies are more biased?


In [ ]:
mass_cuts = [1e9, 1e10, 10**10.5, 1e11]

k_lin  = np.logspace(-2, 1, 500)
Pk_lin = linear_pk_eh98(k_lin)

# YOUR CODE HERE
# Part A: for each mass cut, compute P(k) and plot.
# Part B: compute P(k) separately for star-forming and quiescent galaxies and plot.
# In a second panel, plot b(k) = sqrt(P_gal / P_lin) for all samples.
# Print the large-scale bias (k < 0.1 h/Mpc) for each selection.


## Optional — Cloud-In-Cell (CIC) and Triangular Shape Cloud (TSC) schemes

The NGP window suppresses power at small scales through convolution with a top-hat kernel
in real space. Higher-order schemes produce smoother windows and extend the reliable range
of $P(k)$ toward the Nyquist frequency.

The window functions in Fourier space are:
$$W_{\rm NGP}(k_j) = \mathrm{sinc}\!\left(\frac{k_j \Delta x}{2\pi}\right), \quad
  W_{\rm CIC}(k_j) = \mathrm{sinc}^2\!\left(\frac{k_j \Delta x}{2\pi}\right), \quad
  W_{\rm TSC}(k_j) = \mathrm{sinc}^3\!\left(\frac{k_j \Delta x}{2\pi}\right)$$

To deconvolve, divide $\tilde{\delta}(\mathbf{k})$ by $\prod_{j=x,y,z}W(k_j)$
before squaring.

Tasks:
- Implement `cic_assign` (trilinear weighting to 8 cells) and `tsc_assign`
  (quadratic kernel over 27 cells).
- Compute $P(k)$ with all three schemes on the same galaxy sample; compare on a single plot.
- At which $k$ do the three estimates diverge? How does the Nyquist suppression differ?
- Implement the deconvolution and verify that all three agree up to higher $k$.


In [ ]:
def cic_assign(positions, box_size, n_grid):
    """
    Cloud-In-Cell: assign each galaxy to the 8 surrounding cells
    with weights proportional to the trilinear overlap volume.
    """
    # YOUR CODE HERE
    pass


def tsc_assign(positions, box_size, n_grid):
    """
    Triangular Shape Cloud: use the quadratic kernel
    w(d) = 0.75 - d^2  for |d| < 0.5,
           0.5*(1.5-|d|)^2  for 0.5 <= |d| < 1.5
    applied over 27 neighbouring cells.
    """
    # YOUR CODE HERE
    pass


# YOUR CODE HERE
# Compute P(k) with NGP, CIC, and TSC; plot them together.
# Plot the ratio P_MAS / P_NGP to quantify the window-function suppression.
